# Замер цены рецепта на видеокарте

Этот прогон ничего не обучает насовсем. Он меряет две вещи, из которых
складывается вся стоимость рецепта.

1. Скорость кодирования корпуса. Оценка после каждого этапа это прогон
   всех 62 594 фрагментов, и это самая частая операция в рецепте.
2. Секунду на шаг обучения для каждого состава. Этапы отличаются именно
   составом, а не чем-то ещё.

Дальше цена каждого этапа получается умножением, и печатается таблица
с итогом в часах. Это ответ на вопрос, влезет ли рецепт в остаток квоты,
полученный **до** того, как квота потрачена.

Правило проекта, за нарушение которого уже заплачено сутками. Объём
назначается по замеру, а не по арифметике.

**Около двадцати минут на карте.**

In [ ]:
import os
for ключ in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE'):
    os.environ[ключ] = '1'
# Одна карта. Библиотека обучения сама заворачивает модель
# в DataParallel, как только видит две, а он и зависал, и падал
# по памяти. Прятать вторую надёжнее, чем просить не трогать.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

ФРАГМЕНТОВ_НА_ЗАМЕР = 4096   # на скольких мерить скорость кодирования
ШАГОВ_НА_ЗАМЕР = 6           # сколько шагов обучения мерить на каждый состав

import time
НАЧАЛО = time.time()


def прошло():
    с = time.time() - НАЧАЛО
    return f'{с // 60:.0f} мин {с % 60:02.0f} с'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch, transformers, sentence_transformers
print('transformers', transformers.__version__)
print('sentence-transformers', sentence_transformers.__version__)
print('карт:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'нет карты: Settings -> Accelerator'

In [ ]:
import glob, shutil, zipfile


def find_source(root='/kaggle/input', max_depth=5):
    found, archives = [], []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs:
            found.append((1 if '__output__.json' in files else 0, cur))
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    if found:
        found.sort()
        return found[0][1], None
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


src, archive = find_source()
assert src, 'датасет с кодом не подключён'
if archive:
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for имя in os.listdir(src):
        путь, цель = os.path.join(src, имя), os.path.join('/kaggle/working', имя)
        if os.path.isdir(путь):
            shutil.copytree(путь, цель, dirs_exist_ok=True)
        elif имя.endswith(('.py', '.md')):
            shutil.copyfile(путь, цель)

os.makedirs('/kaggle/working/chunks', exist_ok=True)
dst = '/kaggle/working/chunks/base.jsonl'
if os.path.exists(dst):
    os.remove(dst)
найдено = (glob.glob('/kaggle/input/**/chunks_base.jsonl', recursive=True)
           + glob.glob('/kaggle/input/**/chunks_base.jsonl.gz', recursive=True))
assert найдено, 'нет chunks_base.jsonl'
import gzip
opener = gzip.open if найдено[0].endswith('.gz') else open
with opener(найдено[0], 'rb') as fi, open(dst, 'wb') as fo:
    shutil.copyfileobj(fi, fo, length=1 << 20)
n = sum(1 for _ in open(dst, encoding='utf-8'))
assert n == 62594, f'ожидалось 62594, а не {n}'
print('нарезка:', n)

подготовка = glob.glob('/kaggle/input/**/train_prepared.jsonl', recursive=True)
assert подготовка, ('нет train_prepared.jsonl: подключите вывод ядра подготовки '
                    'или датасет с ним')
print('подготовка:', подготовка[0])
print('прошло', прошло())

In [ ]:
import subprocess

cmd = ['python', 'run_phase_d.py', '--step', 'замер',
       '--prepared', подготовка[0],
       '--probe-chunks', str(ФРАГМЕНТОВ_НА_ЗАМЕР),
       '--probe-steps', str(ШАГОВ_НА_ЗАМЕР),
       '--weights', '/kaggle/working/zamer/weights',
       '--journal', '/kaggle/working/zamer/journal.json',
       '--runs', '/kaggle/working/zamer']
print('запуск:', ' '.join(cmd), flush=True)
код = subprocess.run(cmd, cwd='/kaggle/working').returncode
print('\nкод возврата:', код, '| прошло', прошло())
assert код == 0, f'замер вернул {код}'

In [ ]:
import json, shutil
r = json.load(open('/kaggle/working/zamer/zamer_recepta.json', encoding='utf-8'))
print('кодирование:', r['кодирование, фрагм./с'], 'фрагм./с')
print('одна оценка корпуса:', r['оценка корпуса, мин'], 'мин')
print()
for э in r['этапы']:
    print(f"{э['метка']:<16} {э['всего, мин']:>8} мин")
print(f"\nИТОГО {r['итого часов']} ч на весь рецепт")

shutil.rmtree('/kaggle/working/chunks', ignore_errors=True)
shutil.rmtree('/kaggle/working/zamer/weights', ignore_errors=True)
print('\nвсего', прошло())